# DynaPrompt Per-Token Analysis

**Setup:** Runtime  Change runtime type  T4 GPU  Save

In [ ]:
!nvidia-smi

## Clone Repository

In [ ]:
import os
os.chdir('/content')

!rm -rf 6694-DynaPrompt
!git clone https://github.com/ch3889/6694-DynaPrompt.git
!cd 6694-DynaPrompt && git checkout zk2295 && git submodule update --init --recursive

os.chdir('/content/6694-DynaPrompt')
print(f'Working directory: {os.getcwd()}')
!ls -la models/stable_diffusion_compvis/ldm/ | head -10

## Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q diffusers==0.21.4 accelerate safetensors huggingface-hub
!pip install -q omegaconf einops pytorch-lightning
!pip install -q Pillow numpy matplotlib tqdm scikit-image
!pip install -q kornia albumentations opencv-python imageio imageio-ffmpeg

## Install Transformers

In [ ]:
import subprocess, sys

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.33.2'], check=True, timeout=120)
    print(' transformers')
except:
    print(' Tokenizers build failed, installing without...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', '--no-deps'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'filelock', 'huggingface-hub', 'packaging', 'pyyaml', 'regex', 'requests', 'tqdm'])
    print(' transformers (no tokenizers)')

## Setup Python Paths (NO pip install needed!)

In [ ]:
import sys
import os

# Add paths to Python's search path
sys.path.insert(0, '/content/6694-DynaPrompt')
sys.path.insert(0, '/content/6694-DynaPrompt/models/stable_diffusion_compvis')
sys.path.insert(0, '/content/6694-DynaPrompt/models/stable_diffusion_compvis/src/taming-transformers')
sys.path.insert(0, '/content/6694-DynaPrompt/models/stable_diffusion_compvis/src/clip')

# Verify paths
print('Python paths configured:')
for p in sys.path[:5]:
    print(f'  {p}')

# Test import
try:
    import ldm
    print('\n ldm module imported successfully!')
    print(f'  Location: {ldm.__file__}')
except ImportError as e:
    print(f'\n Error: {e}')
    !ls -la /content/6694-DynaPrompt/models/stable_diffusion_compvis/

## Load CLIP

In [ ]:
from dynaprompt.wrapper import DynaPromptPipeline
import torch

print("Initializing DynaPrompt with per-token analysis...")
dynaprompt = DynaPromptPipeline(
    config_path="/content/6694-DynaPrompt/configs/dynaprompt_config.yaml",
    ckpt_path=checkpoint_path,
    device="cuda"
)
print("✓ DynaPrompt ready!")

prompt = "a golden retriever playing with a red ball in a snowy park"
print(f"Prompt: \"{prompt}\"")
print("\nGenerating image (2-3 minutes)...")

results = dynaprompt.generate_with_feedback(
    prompt=prompt,
    steps=50,
    cfg_scale=7.5,
    height=512,
    width=512,
    seed=42,
    feedback_enabled=True
)

from PIL import Image
import numpy as np

img = results["images"][0]
if isinstance(img, torch.Tensor):
    img = img.cpu().numpy()
if img.max() <= 1.0:
    img = (img * 255).astype(np.uint8)
Image.fromarray(img)

In [ ]:
from huggingface_hub import hf_hub_download

print('Downloading Stable Diffusion v1.5...')
checkpoint_path = hf_hub_download(
    repo_id='runwayml/stable-diffusion-v1-5',
    filename='v1-5-pruned-emaonly.ckpt',
    cache_dir='/content/models'
)
print(f' Downloaded to: {checkpoint_path}')

## Initialize DynaPrompt

In [ ]:
from dynaprompt.wrapper import DynaPromptWrapper
import torch

print('Initializing DynaPrompt with per-token analysis...')
dynaprompt = DynaPromptWrapper(
    model_path=checkpoint_path,
    device='cuda',
    use_per_token_analysis=True
)
print(' DynaPrompt ready!')

## Generate Test Image

In [ ]:
import matplotlib.pyplot as plt

print("Generating baseline (WITHOUT feedback)...")
results_baseline = dynaprompt.generate_with_feedback(
    prompt=prompt,
    steps=50,
    cfg_scale=7.5,
    height=512,
    width=512,
    seed=42,
    feedback_enabled=False
)

img_baseline = results_baseline["images"][0]
if isinstance(img_baseline, torch.Tensor):
    img_baseline = img_baseline.cpu().numpy()
if img_baseline.max() <= 1.0:
    img_baseline = (img_baseline * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(img_baseline)
axes[0].set_title("WITHOUT DynaPrompt Feedback", fontsize=16, fontweight="bold", color="red")
axes[0].axis("off")

axes[1].imshow(img)
axes[1].set_title("WITH DynaPrompt Feedback", fontsize=16, fontweight="bold", color="green")
axes[1].axis("off")

plt.tight_layout()
plt.show()
print("\n✓ Comparison complete!")

## Run Full Test Suite

In [ ]:
os.chdir('/content/6694-DynaPrompt')
!python test_per_token_analysis.py

## View Results

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt

with open('/content/6694-DynaPrompt/per_token_analysis_results.json') as f:
    results = json.load(f)

for i, r in enumerate(results):
    print(f"\n{'='*70}")
    print(f"Test {i+1}: {r['prompt']}")
    print('='*70)
    
    if r.get('weak_tokens'):
        print('\n Underrepresented concepts detected:')
        for t in r['weak_tokens']:
            print(f'    {t}')
    else:
        print('\n All concepts well represented')
    
    img = Image.open(r['image_path'])
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Test {i+1}: {r['prompt'][:60]}...", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Generate Visualizations

In [ ]:
os.chdir('/content/6694-DynaPrompt')
!python visualize_token_analysis.py

from IPython.display import Image as IPImage

viz_files = {
    'clip_scores_evolution.png': ' CLIP Score Evolution',
    'weak_tokens_frequency.png': ' Weak Token Frequency',
    'weak_tokens_timeline.png': ' Weak Token Timeline'
}

for filename, title in viz_files.items():
    print(f'\n{title}:')
    display(IPImage(f'/content/6694-DynaPrompt/{filename}'))

## Baseline Comparison

In [ ]:
import matplotlib.pyplot as plt

print('Generating baseline (WITHOUT per-token analysis)...')
dynaprompt_baseline = DynaPromptWrapper(
    model_path=checkpoint_path,
    device='cuda',
    use_per_token_analysis=False
)

image_baseline = dynaprompt_baseline.generate(
    prompt=prompt,
    num_inference_steps=50,
    guidance_scale=7.5,
    feedback_frequency=5,
    alpha=0.05
)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(image_baseline)
axes[0].set_title('WITHOUT Per-Token Analysis\n(Standard DynaPrompt)', fontsize=16, fontweight='bold', color='red')
axes[0].axis('off')

axes[1].imshow(image)
axes[1].set_title('WITH Per-Token Analysis\n(Enhanced)', fontsize=16, fontweight='bold', color='green')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print('\n Comparison complete!')

## Download Results

In [ ]:
os.chdir('/content/6694-DynaPrompt')

!zip -qr dynaprompt_results.zip *.png *.json per_token_test_*.png 2>/dev/null
!ls -lh dynaprompt_results.zip

from google.colab import files
files.download('dynaprompt_results.zip')
print('\n All results downloaded!')